In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import numpy as np
import pandas as pd


RAW_PATH = "diabetic_data.csv"
CLEANED_PATH = "diabetic_data_cleaned.csv"

pd.set_option("display.max_columns", 60)

## 1. Load the raw dataset

In [ ]:
df = pd.read_csv(RAW_PATH)
print(f"Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Explore the data (EDA)

In [ ]:
df.info()

### Target variable: `readmitted`

In [ ]:
print(df["readmitted"].value_counts())
print()
print(df["readmitted"].value_counts(normalize=True).round(3))

The target is **imbalanced** (`NO` ~54%, `>30` ~35%, `<30` ~11%). This matters for Milestone 2 model training (e.g. class weighting, resampling).

### Missing values (this dataset marks missing data as `'?'`, not NaN)

In [ ]:
qmark_counts = (df == "?").sum().sort_values(ascending=False)
qmark_counts = qmark_counts[qmark_counts > 0]
missing_pct = (100 * qmark_counts / len(df)).round(1)
pd.DataFrame({"missing_count": qmark_counts, "missing_pct": missing_pct})

### Duplicate patients

In [ ]:
dup = df["patient_nbr"].duplicated().sum()
print(f"{dup} repeat encounters from returning patients")

## 3. Clean the data

Steps:
- Replace `'?'` with proper `NaN`
- Drop columns with too much missingness (`weight` ~97%, `medical_specialty` ~49%, `payer_code` ~40%)
- Drop rows missing `race` (only ~2.2%, safe to drop)
- Remove expired/hospice discharges (can't be "readmitted", so they're noise in the target)
- Keep only the first encounter per patient (avoids leakage from repeat visits later)

In [ ]:
df_clean = df.copy()

# Replace '?' with NaN
df_clean.replace("?", np.nan, inplace=True)

# Drop high-missingness columns
cols_to_drop = ["weight", "medical_specialty", "payer_code"]
df_clean.drop(columns=cols_to_drop, inplace=True)
print(f"Dropped columns: {cols_to_drop}")

In [ ]:
before = len(df_clean)
df_clean.dropna(subset=["race"], inplace=True)
print(f"Dropped {before - len(df_clean)} rows with missing race")

In [ ]:
# discharge_disposition_id codes for death/hospice — can't be "readmitted"
expired_codes = [11, 13, 14, 19, 20, 21]

before = len(df_clean)
df_clean = df_clean[~df_clean["discharge_disposition_id"].isin(expired_codes)]
print(f"Removed {before - len(df_clean)} rows for expired/hospice discharges")

In [ ]:
before = len(df_clean)
df_clean = (
    df_clean.sort_values("encounter_id")
    .drop_duplicates(subset="patient_nbr", keep="first")
)
print(f"Reduced to {len(df_clean)} unique patients (removed {before - len(df_clean)} repeat encounters)")

In [ ]:
print(f"Final cleaned shape: {df_clean.shape}")
df_clean.head()

## 4. Save the cleaned dataset

In [ ]:
df_clean.to_csv(CLEANED_PATH, index=False)
print(f"Saved cleaned dataset to: {CLEANED_PATH}")

## Summary

- Raw dataset: 101,766 rows x 50 columns
- Cleaned dataset: rows/columns after dropping sparse columns, missing-race rows, expired/hospice discharges, and duplicate patients
- `readmitted` target is imbalanced — flag this for the Milestone 2 modeling team
- `diag_1`, `diag_2`, `diag_3` (ICD-9 codes) still need grouping/encoding — that's a Milestone 2 feature engineering task, not done here

**Next step (Milestone 2):** feature engineering + training the readmission risk prediction model on `diabetic_data_cleaned.csv`.